In [7]:
import sqlite3
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go

In [ ]:
def analysis(df):
    # Create figure
    fig = go.Figure(data=[
        go.Candlestick(
            x=df['timestamp'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='DOGE/USD'
        ),
        go.Scatter(
            x=df['timestamp'],
            y=df['MA7'],
            line=dict(color='orange', width=1.5),
            name='MA7'
        ),
        go.Scatter(
            x=df['timestamp'],
            y=df['MA25'],
            line=dict(color='blue', width=1.5),
            name='MA25'
        ),
        go.Scatter(
            x=df['timestamp'],
            y=df['MA99'],
            line=dict(color='purple', width=2, dash='dot'),
            name='MA99'
        ),
        go.Scatter(
            x=df[df['buy']]['timestamp'],
            y=df[df['buy']]['close'],
            mode='markers',
            marker_symbol='triangle-up',
            marker_color='lime',
            marker_size=12,
            name='Buy Signal'
        ),
        go.Scatter(
            x=df[df['sell']]['timestamp'],
            y=df[df['sell']]['close'],
            mode='markers',
            marker_symbol='triangle-down',
            marker_color='red',
            marker_size=12,
            name='Sell Signal'
        )
    ])
    
    fig.update_layout(
        title=f'Technical Analysis',
        xaxis_rangeslider_visible=False,
        yaxis_title='Price (USD)',
        hovermode='x unified',
        showlegend=True
    )
    return fig

In [ ]:
def get_kline(symbol, freq):
    conn = sqlite3.connect(f'data/{symbol}.db')
    df = pd.read_sql("SELECT * FROM klines ORDER BY start_t", conn)
    df.index = pd.to_datetime(df['start_t'], unit='ms')
    conn.close()
    if freq == 1:
        return df
    # 转换时间戳为datetime并创建小时分组
    # 定义聚合规则
    agg_funcs = {
        'start_t': 'min', 'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum', 'amount': 'sum', 'trade_cnt': 'sum',
        'taker_vol': 'sum', 'taker_amt': 'sum', 'end_t': 'max'
    }
    # 执行聚合
    df = df.resample(f'{freq}min').agg(agg_funcs)
    return df

In [14]:
df = get_kline('DOGEUSD_PERP', 60)
for n in [7]:
    df[f'MA{n}'] = df.close.rolling(n).mean()
analysis(df).show()

KeyError: 'timestamp'

,start_t,open,high,low,close,volume,amount,trade_cnt,taker_vol,taker_amt,end_t
start_t,,,,,,,,,,,
2025-01-01 00:00:00,1735689600000,0.31550,0.32106,0.31512,0.32071,783735.0,2.461181e+07,6555,450900.0,1.415522e+07,1735693199999
2025-01-01 01:00:00,1735693200000,0.32080,0.32080,0.31766,0.31799,508975.0,1.595462e+07,8546,213887.0,6.705735e+06,1735696799999
2025-01-01 02:00:00,1735696800000,0.31797,0.31951,0.31775,0.31951,441702.0,1.386212e+07,6271,291501.0,9.148293e+06,1735700399999
2025-01-01 03:00:00,1735700400000,0.31951,0.31961,0.31805,0.31845,190896.0,5.989968e+06,2300,92708.0,2.909630e+06,1735703999999
2025-01-01 04:00:00,1735704000000,0.31842,0.31866,0.31566,0.31572,406369.0,1.282010e+07,4072,156170.0,4.927770e+06,1735707599999
...,...,...,...,...,...,...,...,...,...,...,...
2025-03-14 19:00:00,1741978800000,0.17110,0.17235,0.17021,0.17084,219770.0,1.282175e+07,3093,112940.0,6.585137e+06,1741982399999
2025-03-14 20:00:00,1741982400000,0.17086,0.17088,0.16918,0.17048,219177.0,1.290056e+07,3630,88058.0,5.180249e+06,1741985999999
2025-03-14 21:00:00,1741986000000,0.17041,0.17167,0.16997,0.17153,132075.0,7.738467e+06,1950,67512.0,3.953990e+06,1741989599999
